In [32]:
import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, Model, Input
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

# reproducibility
seed = 42
os.environ['PYTHONHASHSEED'] = str(seed)
np.random.seed(seed)
random.seed(seed)
tf.random.set_seed(seed)


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
csv_path = "/content/drive/MyDrive/Deep Learning/ECG_10s_2500.csv"

df = pd.read_csv(csv_path)
print("Data shape:", df.shape)

Data shape: (45630, 2505)


In [42]:
# F1 metric (used for reporting in Keras; for thresholded F1 we'll use sklearn later)
def f1_keras(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred > 0.5, tf.float32)

    tp = tf.reduce_sum(y_true * y_pred)
    fp = tf.reduce_sum((1.0 - y_true) * y_pred)
    fn = tf.reduce_sum(y_true * (1.0 - y_pred))

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2.0 * precision * recall / (precision + recall + 1e-8)
    return f1

# Focal loss (works well with class imbalance)
def focal_loss(gamma=2.0, alpha=0.75):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        eps = 1e-7
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        bce = - (y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
        pt = tf.where(tf.equal(y_true, 1), y_pred, 1 - y_pred)
        loss = alpha * tf.pow(1.0 - pt, gamma) * bce
        return tf.reduce_mean(loss)
    return loss

# Attention layer: variables created in __init__ (no TF variable creation in call)
class SafeAttention(layers.Layer):
    def __init__(self, units):
        super(SafeAttention, self).__init__()
        self.W = layers.Dense(units)
        self.V = layers.Dense(1)

    def call(self, inputs):
        # inputs: (batch, time, features)
        score = self.V(tf.nn.tanh(self.W(inputs)))  # (batch, time, 1)
        weights = tf.nn.softmax(score, axis=1)      # (batch, time, 1)
        context = tf.reduce_sum(weights * inputs, axis=1)
        return context


In [34]:
def random_stretch(x, max_stretch=0.08):
    factor = 1.0 + np.random.uniform(-max_stretch, max_stretch)
    old_idx = np.linspace(0, 1, len(x))
    new_idx = np.linspace(0, 1, int(len(x) * factor))
    warped = np.interp(new_idx, old_idx, x)
    if len(warped) > len(x):
        return warped[:len(x)]
    else:
        return np.pad(warped, (0, len(x) - len(warped)), 'constant')

def augment_signal_1d(x_1d,
                      scale_range=(0.9, 1.1),
                      noise_std_factor=0.02,
                      max_shift=100,
                      stretch_prob=0.5,
                      cutout_prob=0.3):
    x = x_1d.copy()

    # amplitude scaling
    if np.random.rand() < 0.7:
        scale = np.random.uniform(*scale_range)
        x = x * scale

    # jitter (Gaussian noise)
    if np.random.rand() < 0.7:
        std = x.std()
        if std < 1e-8:
            std = 1.0
        noise = np.random.normal(0, noise_std_factor * std, size=x.shape)
        x = x + noise

    # small time shift with zero padding
    if np.random.rand() < 0.7:
        shift = np.random.randint(-max_shift, max_shift + 1)
        if shift > 0:
            x = np.concatenate([x[shift:], np.zeros(shift, dtype=x.dtype)])
        elif shift < 0:
            s = -shift
            x = np.concatenate([np.zeros(s, dtype=x.dtype), x[:-s]])

    # time stretch
    if np.random.rand() < stretch_prob:
        x = random_stretch(x)

    # random cutout
    if np.random.rand() < cutout_prob:
        L = len(x)
        cut_len = int(L * np.random.uniform(0.01, 0.05))
        if cut_len > 0 and L - cut_len > 0:
            start = np.random.randint(0, L - cut_len + 1)
            x[start:start+cut_len] = 0.0

    return x


In [35]:
label_candidates = ['label', 'Label', 'hypo_label', 'y']
pid_candidates   = ['patient_id', 'PatientID', 'patient', 'pid']

label_col = next((c for c in label_candidates if c in df.columns), None)
pid_col   = next((c for c in pid_candidates   if c in df.columns), None)

if label_col is None:
    raise ValueError("Could not find label column in df. Rename your label to one of: " + str(label_candidates))

print("Using label column:", label_col)
if pid_col is not None:
    print("Patient column found (not used for this random split):", pid_col)

meta_cols = {label_col}
if pid_col is not None:
    meta_cols.add(pid_col)
meta_cols |= {'day','Day','count','Count','glucose','glucose_value',
              'glucose_values','Glucose','GlucoseValue'}

signal_cols = [c for c in df.columns if c not in meta_cols]
signal_cols = sorted(signal_cols, key=lambda x: int(x) if str(x).isdigit() else x)

print("Number of signal columns:", len(signal_cols))
print("First 10 signal cols:", signal_cols[:10])

X = df[signal_cols].values.astype(np.float32)
y = df[label_col].astype(int).values

# per-sample normalization (z-score)
X_mean = X.mean(axis=1, keepdims=True)
X_std  = X.std(axis=1, keepdims=True) + 1e-8
X = (X - X_mean) / X_std

timesteps = X.shape[1]
X = X.reshape((-1, timesteps, 1))

print("X shape:", X.shape)
print("Label distribution:", np.bincount(y))


Using label column: label
Patient column found (not used for this random split): patient_id
Number of signal columns: 2500
First 10 signal cols: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
X shape: (45630, 2500, 1)
Label distribution: [43169  2461]


In [36]:
indices = np.arange(len(X))

# 70% train, 15% val, 15% test
train_idx, temp_idx = train_test_split(indices, test_size=0.30, stratify=y, random_state=seed)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=y[temp_idx], random_state=seed+1)

X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]

print("Train shape:", X_train.shape, "Positives:", y_train.sum())
print("Val shape:",   X_val.shape,   "Positives:", y_val.sum())
print("Test shape:",  X_test.shape,  "Positives:", y_test.sum())
print("Train pos ratio:", y_train.mean())

# Augment positive samples in training set to reach target_pos_ratio
target_pos_ratio = 0.30
pos_mask = (y_train == 1)
neg_mask = (y_train == 0)

X_pos = X_train[pos_mask]
X_neg = X_train[neg_mask]

n_pos = len(X_pos)
n_neg = len(X_neg)
print("Before aug - train positives:", n_pos, "negatives:", n_neg)

# how many extra positives needed
m_needed = int(max(0, (target_pos_ratio / (1 - target_pos_ratio)) * n_neg - n_pos))
print("Extra augmented positives needed:", m_needed)

aug_signals = []
if m_needed > 0 and n_pos > 0:
    reps_per_pos = int(np.ceil(m_needed / n_pos))
    flat_pos = X_pos[..., 0]  # (n_pos, timesteps)
    added = 0
    for i in range(n_pos):
        x = flat_pos[i]
        for _ in range(reps_per_pos):
            if added >= m_needed:
                break
            x_aug = augment_signal_1d(x)
            aug_signals.append(x_aug)
            added += 1
        if added >= m_needed:
            break

if len(aug_signals) > 0:
    X_aug_pos = np.stack(aug_signals, axis=0)[..., np.newaxis]
    y_aug_pos = np.ones((X_aug_pos.shape[0],), dtype=int)
    X_train_aug = np.concatenate([X_train, X_aug_pos], axis=0)
    y_train_aug = np.concatenate([y_train, y_aug_pos], axis=0)
else:
    X_train_aug = X_train.copy()
    y_train_aug = y_train.copy()

# shuffle augmented train
perm = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]

print("After aug - train shape:", X_train_aug.shape, "Positives:", y_train_aug.sum(), "Pos ratio:", y_train_aug.mean())


Train shape: (31941, 2500, 1) Positives: 1723
Val shape: (6844, 2500, 1) Positives: 369
Test shape: (6845, 2500, 1) Positives: 369
Train pos ratio: 0.05394320778936164
Before aug - train positives: 1723 negatives: 30218
Extra augmented positives needed: 11227
After aug - train shape: (43168, 2500, 1) Positives: 12950 Pos ratio: 0.29999073387694586


In [45]:
def build_cnn_bilstm_attention(timesteps, channels=1, lr=1e-3):
    inp = Input(shape=(timesteps, channels))

    x = layers.Conv1D(64, kernel_size=9, strides=2, padding='same')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(128, kernel_size=7, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    x = layers.Conv1D(256, kernel_size=5, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.Bidirectional(layers.LSTM(256, return_sequences=True, dropout=0.25, recurrent_dropout=0.1))(x)

    x = SafeAttention(128)(x)  # attention reduces to (batch, features)

    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs=inp, outputs=out)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss=focal_loss(gamma=2.0, alpha=0.75),
        metrics=[tf.keras.metrics.AUC(name='auc'), tf.keras.metrics.Recall(name='recall'), tf.keras.metrics.Precision(name='precision'), f1_keras]
    )
    return model

model = build_cnn_bilstm_attention(timesteps)
model.summary()


Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 2500, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_17 (Conv1D)              │ (None, 1250, 64)       │           640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 1250, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_17 (ReLU)                 │ (None, 1250, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_14 (MaxPooling1D) │ (None, 625, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_18 (Conv1D)              │ (None, 625, 128)       │        57,472 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_18          │ (None, 625, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_18 (ReLU)                 │ (None, 625, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_15 (MaxPooling1D) │ (None, 312, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_19 (Conv1D)              │ (None, 312, 256)       │       164,096 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_19          │ (None, 312, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_19 (ReLU)                 │ (None, 312, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_7 (Bidirectional) │ (None, 312, 512)       │     1,050,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ safe_attention (SafeAttention)  │ (None, 512)            │        65,793 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_21 (Dense)                │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_22 (Dense)                │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_23 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,414,402 (5.40 MB)

 Trainable params: 1,413,506 (5.39 MB)

 Non-trainable params: 896 (3.50 KB)

In [ ]:
# Make sure y_train_aug is numpy array
y_train_aug = np.array(y_train_aug)

# compute class weights on augmented dataset
unique_classes = np.unique(y_train_aug)
cw = compute_class_weight(class_weight='balanced', classes=unique_classes, y=y_train_aug)
# map correctly to labels (in case classes are not [0,1] ordering)
class_weights = {int(cls): float(w) for cls, w in zip(unique_classes, cw)}
print("Class weights:", class_weights)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc',
    mode='max',
    patience=12,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_auc',
    factor=0.5,
    patience=4,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train_aug, y_train_aug,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=128,
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weights,
    verbose=1
)


Class weights: {0: 0.7142762591832682, 1: 1.6667181467181467}
Epoch 1/50
338/338 ━━━━━━━━━━━━━━━━━━━━ 671s 2s/step - auc: 0.8376 - f1_keras: 0.2345 - loss: 0.0812 - precision: 0.7330 - recall: 0.5239 - val_auc: 0.7685 - val_f1_keras: 0.0217 - val_loss: 0.0473 - val_precision: 0.2897 - val_recall: 0.0840 - learning_rate: 0.0010
Epoch 2/50
338/338 ━━━━━━━━━━━━━━━━━━━━ 658s 2s/step - auc: 0.9436 - f1_keras: 0.2579 - loss: 0.0490 - precision: 0.9273 - recall: 0.7153 - val_auc: 0.8543 - val_f1_keras: 0.0344 - val_loss: 0.0421 - val_precision: 0.3897 - val_recall: 0.2060 - learning_rate: 0.0010
Epoch 3/50
338/338 ━━━━━━━━━━━━━━━━━━━━ 692s 2s/step - auc: 0.9605 - f1_keras: 0.2714 - loss: 0.0411 - precision: 0.9289 - recall: 0.7840 - val_auc: 0.8836 - val_f1_keras: 0.0151 - val_loss: 0.0315 - val_precision: 0.5143 - val_recall: 0.0976 - learning_rate: 0.0010
Epoch 4/50
338/338 ━━━━━━━━━━━━━━━━━━━━ 654s 2s/step - auc: 0.9683 - f1_keras: 0.2735 - loss: 0.0369 - precision: 0.9423 - recall: 0.8058